# 01. Pipeline walkthrough (T03)

Detection -> 5 landmarks -> aligned 112x112 crop -> 512-d ArcFace embedding -> cosine similarity.

**Before running:** put 6 photos in `research/data/photos/` named `<person>_<n>.jpg`, for example `me_1.jpg`, `me_2.jpg`, `me_3.jpg`, `friend_1.jpg`, `friend_2.jpg`, `friend_3.jpg`. Only use photos you have the right to use (your friend must consent). `data/` is git-ignored, so the photos are never committed.

In [ ]:
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
from insightface.app import FaceAnalysis
from insightface.utils import face_align

PHOTO_DIR = Path("../data/photos")  # notebooks run from research/notebooks
app = FaceAnalysis(name="buffalo_l", allowed_modules=["detection", "recognition"],
                   providers=["CPUExecutionProvider"])
app.prepare(ctx_id=-1, det_size=(480, 480))
print("models loaded")

## 1. Load photos and detect the largest face in each

In [ ]:
paths = sorted(p for p in PHOTO_DIR.glob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png"})
print(len(paths), "photos found")
assert len(paths) >= 2, f"Put photos in {PHOTO_DIR.resolve()}"

items = []
for p in paths:
    img = cv2.imread(str(p))
    if img is None:
        print("cannot read", p.name)
        continue
    faces = app.get(img)
    if not faces:
        print("NO FACE FOUND in", p.name)
        continue
    face = max(faces, key=lambda f: (f.bbox[2] - f.bbox[0]) * (f.bbox[3] - f.bbox[1]))
    items.append({"name": p.stem, "person": p.stem.split("_")[0], "img": img, "face": face})
print(len(items), "photos with a detected face")

## 2. Boxes and the 5 landmarks (eyes, nose, mouth corners)

In [ ]:
def show_grid(images, titles, cols=3, size=4):
    rows = (len(images) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * size, rows * size))
    for ax in np.atleast_1d(axes).ravel():
        ax.axis("off")
    for ax, im, t in zip(np.atleast_1d(axes).ravel(), images, titles):
        ax.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB))
        ax.set_title(t, fontsize=10)
    plt.tight_layout()
    plt.show()


annotated = []
for it in items:
    im = it["img"].copy()
    x1, y1, x2, y2 = it["face"].bbox.astype(int)
    thick = max(2, im.shape[1] // 300)
    cv2.rectangle(im, (x1, y1), (x2, y2), (0, 255, 0), thick)
    for (x, y) in it["face"].kps.astype(int):
        cv2.circle(im, (x, y), thick * 2, (0, 0, 255), -1)
    annotated.append(im)
show_grid(annotated, [it["name"] for it in items])

## 3. Aligned 112x112 crops (this is exactly what ArcFace sees)

In [ ]:
crops = [
    face_align.norm_crop(it["img"], landmark=it["face"].kps, image_size=112) for it in items
]
show_grid(crops, [it["name"] for it in items], cols=6, size=2)

## 4. Embeddings and the cosine-similarity matrix

In [ ]:
E = np.stack([it["face"].normed_embedding for it in items])  # unit-length, 512-d
print("embedding matrix:", E.shape, "row norms ~", np.linalg.norm(E, axis=1).round(3))
S = E @ E.T  # cosine similarity because rows are unit vectors

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(S, cmap="viridis", vmin=0, vmax=1)
names = [it["name"] for it in items]
ax.set_xticks(range(len(names)), names, rotation=90)
ax.set_yticks(range(len(names)), names)
for i in range(len(names)):
    for j in range(len(names)):
        ax.text(j, i, f"{S[i, j]:.2f}", ha="center", va="center", color="w", fontsize=8)
fig.colorbar(im)
ax.set_title("Cosine similarity")
plt.tight_layout()
plt.show()

## 5. Check: same-person scores should be clearly higher than cross-person scores

In [ ]:
same, cross = [], []
for i in range(len(items)):
    for j in range(i + 1, len(items)):
        (same if items[i]["person"] == items[j]["person"] else cross).append(S[i, j])
print(f"same-person : n={len(same)}, mean={np.mean(same):.3f}, min={np.min(same):.3f}")
print(f"cross-person: n={len(cross)}, mean={np.mean(cross):.3f}, max={np.max(cross):.3f}")
print("PASS" if np.min(same) > np.max(cross) else "CHECK: scores overlap, inspect the photos")